# Study 8

Compared accurate point, conservative point, and range estimates for short and long GPS trips using skewed travel-time outcomes.

In [1]:
# Import packages and locate the data folder.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binomtest, t
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/Study 1.xlsx').exists())
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache/matplotlib')
ITEMS = ['sb_like', 'sb_info', 'sb_useful', 'sb_accurate', 'sb_trust']
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [2]:
# Load data, apply exclusions, and calculate satisfaction.
d = pd.read_excel(ROOT / 'data/raw/Study 8.xlsx')
raw_n = len(d)

d = d.loc[(d.ExcludeDuplicateIP != 1)].copy()
assert len(d) == 888
d['satisfaction'] = d[ITEMS].mean(axis=1)
rows, descriptions = [], []
display(pd.DataFrame([{'raw_n':raw_n, 'replicated_n':len(d), 'paper_n':888, 'match':len(d)==888}]))

,raw_n,replicated_n,paper_n,match
0,894,888,888,True


In [3]:
# Define helpers for regressions and paper comparisons.
def record(study, label, fit, term, target_b=None, target_se=None, sign=1):
    ci = np.sort(np.asarray(fit.conf_int().loc[term]) * sign)
    b, se = (sign * fit.params[term], fit.bse[term])
    row = {'study': study, 'analysis': label, 'n': int(fit.nobs), 'b': b, 'se': se, 'p': fit.pvalues[term], 'ci_low': ci[0], 'ci_high': ci[1], 'published_b': target_b, 'published_se': target_se, 'b_matches_2dp': None if target_b is None else abs(b - target_b) <= 0.0050001, 'se_matches_2dp': None if target_se is None else abs(se - target_se) <= 0.0050001}
    rows.append(row)

def ols(d, y, terms):
    return sm.OLS(d[y], sm.add_constant(d[terms].astype(float)), missing='drop').fit()

def contrast(d, study, labels, treatment, reference, target_b, target_se, label=None, covariates=()):
    frame = d.copy()
    for name, code in labels.items():
        frame[name] = (frame.Condition == code).astype(float)
    terms = [name for name in labels if name != reference] + list(covariates)
    record(study, label or f'{treatment} - {reference}', ols(frame, 'satisfaction', terms), treatment, target_b, target_se)

def describe(d, study, group='all'):
    for cond, row in d.groupby('Condition').satisfaction.agg(['count', 'mean', 'std', 'sem']).iterrows():
        descriptions.append({'study': study, 'group': group, 'condition': cond, 'n': int(row['count']), 'mean': row['mean'], 'sd': row['std'], 'se_mean': row['sem'], 'ci_low': row['mean'] - t.ppf(0.975, row['count'] - 1) * row['sem'], 'ci_high': row['mean'] + t.ppf(0.975, row['count'] - 1) * row['sem']})

In [4]:
# Compare estimate formats for short trips, long trips, and the pooled sample.
describe(d, 8)
d['long_trip'] = (d.Length == 2).astype(float)
labels = {'point': 1, 'conservative': 2, 'range': 3}
targets = {'short': [(-0.3, 0.14), (0.53, 0.14), (0.83, 0.14)], 'long': [(-0.43, 0.13), (0.25, 0.13), (0.68, 0.13)], 'all': [(-0.37, 0.09), (0.39, 0.1), (0.75, 0.09)]}
for group, frame in [('short', d[d.Length == 1]), ('long', d[d.Length == 2]), ('all', d)]:
    if group != 'all':
        describe(frame, 8, group)
    for (a, b), (beta, se) in zip([('conservative', 'point'), ('range', 'point'), ('range', 'conservative')], targets[group]):
        contrast(frame, 8, labels, a, b, beta, se, f'{group}: {a} - {b}', ('long_trip',) if group == 'all' else ())

In [5]:
# Display estimates beside the paper and check agreement at reported precision.
results = pd.DataFrame(rows)
checked = results.published_b.notna()
assert results.loc[checked, 'b_matches_2dp'].all()
assert results.loc[checked, 'se_matches_2dp'].all()
display(results[['analysis','n','b','published_b','se','published_se','p','b_matches_2dp','se_matches_2dp']])

,analysis,n,b,published_b,se,published_se,p,b_matches_2dp,se_matches_2dp
0,short: conservative - point,445,-0.3005,-0.30,0.1367,0.14,2.8482e-02,True,True
1,short: range - point,445,0.5319,0.53,0.1379,0.14,1.3175e-04,True,True
2,short: range - conservative,445,0.8324,0.83,0.1372,0.14,2.8095e-09,True,True
3,long: conservative - point,443,-0.4339,-0.43,0.1315,0.13,1.0494e-03,True,True
4,long: range - point,443,0.2452,0.25,0.1309,0.13,6.1608e-02,True,True
5,long: range - conservative,443,0.6791,0.68,0.1300,0.13,2.6896e-07,True,True
6,all: conservative - point,888,-0.3663,-0.37,0.0949,0.09,1.2194e-04,True,True
7,all: range - point,888,0.3883,0.39,0.0951,0.10,4.8207e-05,True,True
8,all: range - conservative,888,0.7545,0.75,0.0945,0.09,4.3756e-15,True,True


In [6]:
# Compare group means and standard deviations with the published table.
table = pd.DataFrame(descriptions)
published = {('short',1):(5.13,1.21),('short',2):(4.83,1.33),('short',3):(5.66,.97),('long',1):(5.25,1.10),('long',2):(4.82,1.23),('long',3):(5.50,1.03),('all',1):(5.19,1.15),('all',2):(4.82,1.28),('all',3):(5.58,1.01)}
table['estimate'] = table.condition.map({1:'Accurate point',2:'Conservative point',3:'Range'})
table['paper_mean'] = [published[(r.group,r.condition)][0] for r in table.itertuples()]
table['paper_sd'] = [published[(r.group,r.condition)][1] for r in table.itertuples()]
table['match'] = ((table['mean']-table.paper_mean).abs()<=.0050001)&((table.sd-table.paper_sd).abs()<=.0050001)
assert table['match'].all()
display(table[['group','estimate','n','mean','sd','paper_mean','paper_sd','match']])

,group,estimate,n,mean,sd,paper_mean,paper_sd,match
0,all,Accurate point,292,5.1884,1.1548,5.19,1.15,True
1,all,Conservative point,299,4.8221,1.2805,4.82,1.28,True
2,all,Range,297,5.5764,1.0059,5.58,1.01,True
3,short,Accurate point,148,5.1257,1.2104,5.13,1.21,True
4,short,Conservative point,151,4.8252,1.3277,4.83,1.33,True
5,short,Range,146,5.6575,0.9750,5.66,0.97,True
6,long,Accurate point,144,5.2528,1.0952,5.25,1.10,True
7,long,Conservative point,148,4.8189,1.2349,4.82,1.23,True
8,long,Range,151,5.4980,1.0320,5.50,1.03,True
